# 08 — Moteur A : détection d'anomalies (Isolation Forest)
Entraînement non supervisé sur un échantillon pur du train (1/20, étiquette jamais lue).
Seuil fixé sur la validation au même taux d'alerte que le moteur B (0,179 %), puis évaluation
unique sur les tests A et B et comparaison avec le moteur B.

In [ ]:
import gc
import json
import sys
import time
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

pd.set_option("display.max_columns", None)

PROJET = Path("/mnt/c/Users/Salah/Desktop/SecuMon-IA")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), PROJET)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from secumon.ml.models import anomaly, dataset

GOLD = ROOT / "data" / "processed" / "gold" / "gold.parquet"
MODELS_A = ROOT / "data" / "processed" / "models" / "engine_a"
MODELS_B = ROOT / "data" / "processed" / "models" / "engine_b"
MODELS_A.mkdir(parents=True, exist_ok=True)
TMP_LINUX = Path.home() / ".cache" / "secumon-duckdb-tmp"
TMP_LINUX.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET temp_directory = '{TMP_LINUX.as_posix()}'")
con.execute("SET memory_limit = '1GB'")
con.execute("SET threads = 2")
con.execute("SET enable_progress_bar = false")

meta_b = json.loads((MODELS_B / "meta.json").read_text())
print("GOLD existe :", GOLD.exists(), "| moteur B : seuil", round(meta_b["threshold"], 4))

In [ ]:
t0 = time.perf_counter()
train = dataset.load_sample(con, GOLD, "train", every=20)
print(f"Echantillon pur : {len(train):,} transactions "
      f"(fraudes presentes naturellement : {train['is_fraud'].mean():.3%}, non utilisees)")

X_train = anomaly.anomaly_matrix(train)
model_a = anomaly.fit_isolation_forest(X_train, n_jobs=4)
print(f"Entrainement : {time.perf_counter() - t0:.0f} s")

del train, X_train
gc.collect()
joblib.dump(model_a, MODELS_A / "isolation_forest.joblib")
print("Modele sauvegarde :", MODELS_A / "isolation_forest.joblib")

In [ ]:
TAUX_ALERTE_B = 3042 / 1701371          # taux d'alerte du moteur B sur la validation (0,179 %)

t0 = time.perf_counter()
morceaux = []
for part in dataset.iter_split(con, GOLD, "validation", n_parts=4):
    morceaux.append(pd.DataFrame({
        "is_fraud": part["is_fraud"].to_numpy(dtype=bool),
        "score_a": anomaly.anomaly_score(model_a, anomaly.anomaly_matrix(part)),
    }))
    del part
    gc.collect()
val = pd.concat(morceaux, ignore_index=True)
del morceaux

SEUIL_A = float(np.quantile(val["score_a"], 1 - TAUX_ALERTE_B))
alerte = val["score_a"] >= SEUIL_A
tp = int((alerte & val["is_fraud"]).sum())
print(f"Scoring validation : {time.perf_counter() - t0:.0f} s")
print(f"PR-AUC validation : {average_precision_score(val['is_fraud'], val['score_a']):.4f} "
      f"(hasard = {val['is_fraud'].mean():.4f})")
print(f"Seuil A (taux d'alerte {TAUX_ALERTE_B:.3%}) : {SEUIL_A:.4f}")
print(f"  rappel    : {tp / val['is_fraud'].sum():.1%}")
print(f"  precision : {tp / alerte.sum():.1%}   | alertes : {int(alerte.sum()):,}")

(MODELS_A / "meta.json").write_text(json.dumps({
    "threshold": SEUIL_A, "alert_rate": TAUX_ALERTE_B, "columns": anomaly.ANOMALY_COLUMNS,
    "train_sample_every": 20, "n_estimators": 200, "seed": 42,
}, indent=2))
del val
gc.collect()

In [ ]:
t0 = time.perf_counter()
morceaux = []
for split in ["test_a", "test_b"]:
    for part in dataset.iter_split(con, GOLD, split, n_parts=12):
        morceaux.append(pd.DataFrame({
            "tx_id": part["tx_id"].to_numpy(),
            "score_a": anomaly.anomaly_score(model_a, anomaly.anomaly_matrix(part)),
        }))
        del part
        gc.collect()
scores_a = pd.concat(morceaux, ignore_index=True)
del morceaux
scores_a.to_parquet(MODELS_A / "scores_tests.parquet", index=False)

scores = pd.read_parquet(MODELS_B / "scores_tests.parquet").merge(scores_a, on="tx_id", how="inner")
print(f"Scoring : {time.perf_counter() - t0:.0f} s | lignes : {len(scores):,}")

In [ ]:
SEUIL_B = meta_b["threshold"]

def resume(df):
    y = df["is_fraud"].to_numpy(dtype=bool)
    out = {"transactions": len(df), "fraudes": int(y.sum())}
    for m, col, s in [("A", "score_a", SEUIL_A), ("B", "score", SEUIL_B)]:
        p = df[col].to_numpy()
        alerte = p >= s
        tp, n_a = int((alerte & y).sum()), int(alerte.sum())
        out[f"pr_auc_{m}"] = average_precision_score(y, p) if 0 < y.sum() < len(y) else np.nan
        out[f"rappel_{m}"] = tp / y.sum() if y.sum() else np.nan
        out[f"precision_{m}"] = tp / n_a if n_a else np.nan
    return pd.Series(out)

print("=== Global ===")
display(scores.groupby("split")[["is_fraud", "score", "score_a"]].apply(resume).round(4))
print("=== Par zone ===")
display(scores.groupby(["split", "zone"])[["is_fraud", "score", "score_a"]].apply(resume).round(4))

In [ ]:
# --- 1. Complementarite : parmi les fraudes manquees par B, combien A en rattrape ---
fraudes = scores[scores["is_fraud"]].copy()
fraudes["detectees_B"] = fraudes["score"] >= SEUIL_B
fraudes["manquees_B"] = ~fraudes["detectees_B"]
fraudes["rattrapees_par_A"] = fraudes["manquees_B"] & (fraudes["score_a"] >= SEUIL_A)

comp = (fraudes.groupby(["split", "zone"])[["detectees_B", "manquees_B", "rattrapees_par_A"]]
        .sum().astype(int))
comp.insert(0, "fraudes", fraudes.groupby(["split", "zone"]).size())
print("=== Complementarite (fraudes uniquement) ===")
print(comp.reset_index().to_string(index=False))


# --- 2. Cout de la combinaison "A ou B" sur toutes les transactions ---
def union(df):
    y = df["is_fraud"].to_numpy(dtype=bool)
    a = df["score_a"].to_numpy() >= SEUIL_A
    b = df["score"].to_numpy() >= SEUIL_B
    out = {}
    for nom, alerte in [("B seul", b), ("A ou B", a | b)]:
        tp, n_a = int((alerte & y).sum()), int(alerte.sum())
        out[f"rappel {nom}"] = round(tp / y.sum(), 4) if y.sum() else np.nan
        out[f"precision {nom}"] = round(tp / n_a, 4) if n_a else np.nan
        out[f"alertes {nom}"] = n_a
    return pd.Series(out)

print("\n=== B seul contre A ou B ===")
print(scores.groupby("split")[["is_fraud", "score", "score_a"]].apply(union).T.to_string())

## Conclusion — moteur A v1 (Isolation Forest, 16 features comportementales, non supervisé)

Entraînement : 773 521 transactions (échantillon pur, 0,116 % de fraudes non utilisées), 7 s.
Seuil fixé sur la validation au taux d'alerte du moteur B (0,179 %).

| Jeu | PR-AUC A | Rappel A | Précision A | Rappel B (rappel) |
|---|---|---|---|---|
| Validation 2015 | 0,0225 (hasard 0,0019) | 5,2 % | 5,6 % | 81,7 % |
| Test A 2016 | 0,0214 | 3,2 % | 4,3 % | 73,6 % |
| Test B 2017–2020 | 0,0263 | 9,5 % | 5,4 % | 41,1 % |

**Complémentarité** : A rattrape 114 des 2 801 fraudes italiennes manquées par B (4,1 %).
**« A ou B »** : test B +2,4 points de rappel pour des alertes × 3,7 (précision 66 % → 19 %),
soit ~70 fausses alertes par fraude supplémentaire → combinaison simple rejetée.

**Analyse** : les fraudes manquées par B (petits montants, commerçants connus) ne sont pas des
anomalies ; les anomalies du dataset sont surtout des comportements légitimes inhabituels
(voyages, gros achats ponctuels, artefacts de vitesse du générateur).

**Résultat négatif documenté.** Usage retenu : score A comme information d'entrée du moteur de
décision et indicateur de surveillance des dérives, pas comme règle d'alerte.